<a href="https://colab.research.google.com/github/JasmineLee7/movie-rec/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Milestone 2!")

Milestone 2


#Part A: Load the MovieLens ratings

In [ ]:
import os, urllib.request, zipfile, ssl
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    with urllib.request.urlopen(
            "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
            context=ctx) as resp, open("ml-100k.zip", "wb") as f:
        f.write(resp.read())
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"])

print(ratings.head())
print("Number of ratings:", len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


This code loads the MovieLens 100K ratings. It first imports the tools for downloading files, unzipping them, and working with data (NumPy and Pandas). If the `ml-100k` folder doesn't already exist, it downloads and unzips the dataset, so rerunning the notebook doesn't download it again. Then `pd.read_csv` reads the tab-separated ratings file into a table with four columns: user, item (movie), rating, and time. Each row is one known rating, so the data is in long form. The printout shows the first few rows and confirms there are 100,000 ratings.

# Part B: Make and honest train/test split

In [ ]:
def split_ratings(ratings, test_frac=0.10, seed=0):
  # Hide ratings without removing a user's or movie's last training rating.
  rng = np.random.default_rng(seed)
  is_test = np.zeros(len(ratings), dtype=bool)

  user_left = ratings.groupby("user").size().to_dict()
  item_left = ratings.groupby("item").size().to_dict()

  for pos in rng.permutation(len(ratings)):
      row = ratings.iloc[pos]
      user, item = row["user"], row["item"]

      if (rng.random() < test_frac
            and user_left[user] > 1
            and item_left[item] > 1):
        is_test[pos] = True
        user_left[user] -= 1
        item_left[item] -= 1

  return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)
print("train:", len(train), "test:", len(test))
print("test fraction:", round(len(test) / len(ratings), 3))

train: 90032 test: 9968
test fraction: 0.1


This function splits the ratings into training and test sets so the recommendation system can be fairly evaluated. It visits the ratings in a random order, giving each one around a 10% of being place in the test set. It returns train, which are the ratings used to build the recommendation models, and test, which are hidden ratings used to check prediction accuracy.

# Part C: Build the training matrix

In [ ]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
    """Return a user-by-movie matrix; missing ratings are np.nan."""
    R = np.full((N_USERS, N_ITEMS), np.nan)
    R[df["user"].to_numpy() - 1,
      df["item"].to_numpy() - 1] = df["rating"].to_numpy()
    return R

R = to_matrix(train)
known = ~np.isnan(R)

print("Matrix shape:", R.shape)
print("Known training ratings:", known.sum())

Matrix shape: (943, 1682)
Known training ratings: 90032


This code turns the training table into a matrix with 943 user rows and 1,682 movie columns. It starts with every entry set to `np.nan`, meaning unknown, and fills in only the ratings in `train`. We subtract 1 from user and movie IDs because the IDs start at 1 but Python positions start at 0. The Boolean array `known` is True wherever the matrix contains a training rating. Its sum should equal the number of rows in `train`. Test ratings stay missing in this matrix so the models cannot learn from the answers they will be tested on.

In [ ]:
truth = test["rating"].to_numpy()
u = test["user"].to_numpy() - 1
i = test["item"].to_numpy() - 1

This code prepares the hidden ratings for evaluation. `truth` stores the actual test ratings, while `u` and `i` store their user row and movie column positions, adjusted to start at 0. Matching positions in these three arrays refer to the same test example. Later sections will use `u` and `i` to look up predictions and compare them with `truth`. These test ratings are used only for scoring; they are not added to the training matrix.

## Part D: Score Predictions with RMSE

In [ ]:
def rmse(pred, truth):
    """Root mean squared error, measured in stars."""
    return np.sqrt(np.mean((pred - truth)**2))

The accuracy of each recommendation method can be evaluated using RMSE. This function measures the difference between predicted and actual movie ratings. It squares these differences and finds their average before taking the square root. Smaller RMSE values indicate better predictions and help us identify which method performs more accurately.

# Part E: Build Simple Baselines

In [ ]:

global_mean = np.nanmean(R)

user_means = np.nanmean(R, axis=1)
movie_means = np.nanmean(R, axis=0)

global_pred = np.full(len(truth), global_mean)
user_pred = user_means[u]
movie_pred = movie_means[i]

print("Global mean RMSE:", round(rmse(global_pred, truth), 4))
print("User mean RMSE:", round(rmse(user_pred, truth), 4))
print("Movie mean RMSE:", round(rmse(movie_pred, truth), 4))


Global mean RMSE: 1.1349
User mean RMSE: 1.0426
Movie mean RMSE: 1.0284


Based on the results, the movie mean was the most accurate method with an RMSE of 1.0284. The user mean followed with a score of 1.0426 while the global mean had the highest error at 1.1349. This suggests that a movie's average rating is more useful for predicting ratings than relying on individual user averages or the overall average. The results also show how the three methods differ in accuracy.

# Part F: Combine user and movie effects

In [ ]:
def bias_baseline(R):
  # Overall mean plus one user effect and one movie effect.
  known = ~np.isnan(R)
  mu = np.nanmean(R)
  b_user = np.nanmean(R - mu, axis=1)

  residual = R - mu - b_user[:, None]

  n_item = known.sum(axis=0)
  b_item = np.zeros(R.shape[1])
  b_item[has_rating] = (
      np.nansum(residual, axis=0) / n_item[has_rating])

  return mu, b_user, b_item

In [ ]:
mu, b_user, b_item = bias_baseline(R)

pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)

scores['user + movie ffects'] = rmse(pred, truth)
print(pd.Series(scores, name='test RMSE').round(4).sort_values())